In [4]:
# ============================
# 1. Instalação de dependências
# ============================
!apt-get update -y
!apt-get install -y build-essential cmake git pkg-config \
    libgtk-3-dev libavcodec-dev libavformat-dev libswscale-dev \
    libv4l-dev libxvidcore-dev libx264-dev libjpeg-dev libpng-dev libtiff-dev \
    gfortran openexr libatlas-base-dev python3-dev python3-numpy

# ============================
# 2. Clonar repositórios OpenCV
# ============================
!git clone https://github.com/opencv/opencv.git
!git clone https://github.com/opencv/opencv_contrib.git

# ============================
# 3. Compilar OpenCV com utilitários
# ============================
!mkdir -p build && cd build && cmake \
    -DOPENCV_EXTRA_MODULES_PATH=../opencv_contrib/modules \
    -DBUILD_opencv_apps=ON \
    -DBUILD_opencv_traincascade=ON \
    -DBUILD_opencv_createsamples=ON \
    ../opencv && make -j4

# ============================
# 4. Verificar binários
# ============================
!ls build/bin
!./build/bin/opencv_createsamples --help
!./build/bin/opencv_traincascade --help

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
gfortran is already the newest ve

In [5]:
# ============================
# 5. Preparação do dataset
# ============================
import os
from pathlib import Path
import cv2
import opendatasets as od

# Estrutura de diretórios
os.makedirs("haar_training/positives", exist_ok=True)
os.makedirs("haar_training/negatives", exist_ok=True)
os.makedirs("haar_training/data", exist_ok=True)

print("Estrutura criada.")

# Download do dataset
od.download("https://www.kaggle.com/datasets/fareselmenshawii/face-detection-dataset")

# Diretórios do dataset
IMAGE_DIR = Path("./face-detection-dataset/images/train").resolve()
LABEL_DIR = Path("./face-detection-dataset/labels/train").resolve()

# Criação do arquivo positives.txt
output_file = "haar_training/positives.txt"

with open(output_file, "w") as f:
    for img_path in IMAGE_DIR.glob("*.jpg"):
        label_path = LABEL_DIR / (img_path.stem + ".txt")
        if not label_path.exists():
            continue

        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]

        with open(label_path, "r") as lf:
            lines = lf.readlines()

        line_output = str(img_path.resolve())
        line_output += f" {len(lines)}"

        for line in lines:
            _, xc, yc, bw, bh = map(float, line.split())
            x = int((xc - bw/2) * w)
            y = int((yc - bh/2) * h)
            bw = int(bw * w)
            bh = int(bh * h)
            line_output += f" {x} {y} {bw} {bh}"

        f.write(line_output + "\n")

print("positives.txt criado.")

# Criação do arquivo negatives.txt
NEG_IMAGE_DIR = Path("./face-detection-dataset/images/val").resolve()
neg_output = "haar_training/negatives.txt"

with open(neg_output, "w") as f:
    for img_path in NEG_IMAGE_DIR.glob("*.jpg"):
        f.write(str(img_path.resolve()) + "\n")

print("negatives.txt criado.")

Estrutura criada.
Skipping, found downloaded files in "./face-detection-dataset" (use force=True to force download)
positives.txt criado.
negatives.txt criado.


In [12]:
!cd build/bin
!ls | grep opencv_

opencv_contrib


In [18]:
# ============================
# 6. Criação do vetor de amostras positivas
# ============================
!./build/bin/opencv_createsamples \
-info haar_training/positives.txt \
-num 2000 \
-w 24 -h 24 \
-vec haar_training/positives.vec



/bin/bash: line 1: ./build/bin/opencv_createsamples: No such file or directory


In [10]:
# ============================
# 7. Treinamento do classificador Haar Cascade
# ============================
!./build/bin/opencv_traincascade \
-data haar_training/data \
-vec haar_training/positives.vec \
-bg haar_training/negatives.txt \
-numPos 1800 \
-numNeg 500 \
-numStages 5 \
-w 24 -h 24 \
-featureType HAAR \
-minHitRate 0.995 \
-maxFalseAlarmRate 0.5 \
-mode ALL

/bin/bash: line 1: ./build/bin/opencv_traincascade: No such file or directory


In [ ]:
# ============================
# 8. Teste do classificador
# ============================
face_cascade = cv2.CascadeClassifier("haar_training/data/cascade.xml")

test_img_path = next(IMAGE_DIR.glob("*.jpg"))
img = cv2.imread(str(test_img_path))
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)

print(f"Faces detectadas: {len(faces)}")